# Qwen3 with Hugging Face `transformers`

Demo of two ways to run `Qwen/Qwen3-0.6B` on this Intel Mac (CPU):

1. **`pipeline`**: the high-level API (tokenize → generate → decode in one call)
2. **`model.generate`**: the lower-level API with full control over the chat template, sampling, and Qwen3's *thinking* mode

Requires `transformers>=4.51` (Qwen3 support) — see `requirements-intel-mac.txt`.

In [1]:
import torch
import transformers
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

print("torch", torch.__version__, "| transformers", transformers.__version__)

MODEL_ID = "Qwen/Qwen3-0.6B"
# Force CPU: torch may report an MPS device on this Mac, but it's unreliable for these models.
DEVICE = "cpu"

/Users/scottblain/AE_Interview/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.2.2 | transformers 4.51.3


## 1. The `pipeline` API

Pass a list of chat messages and the pipeline applies the chat template for you.
Qwen3 thinks by default; appending `/no_think` to the user message turns that off for this turn.

In [2]:
generator = pipeline(
    "text-generation",
    model=MODEL_ID,
    torch_dtype=torch.float32,
    device=DEVICE,
)

Device set to use cpu


In [6]:
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "Explain the attention mechanism in one sentence. /no_think"},
]

out = generator(messages, max_new_tokens=100)

# With chat input, generated_text is the full conversation; the last message is the reply.
print(out[0]["generated_text"][-1]["content"])

<think>

</think>

The attention mechanism enables the model to focus on the most relevant parts of the input by weighting their contributions based on the similarity between the current context and the context it is trying to learn from.


In [8]:
# Plain-string prompts work too (raw text continuation, no chat template).
out = generator("The three primary colors are", max_new_tokens=30)
print(out[0]["generated_text"])

The three primary colors are red, blue, and yellow. What is the number of different ways that the colors can be arranged in a row of 5 positions, where each


In [9]:
# Batched prompts + sampling parameters
prompts = [
    [{"role": "user", "content": "Give me a name for a pet robot. /no_think"}],
    [{"role": "user", "content": "Write a haiku about the ocean. /no_think"}],
]
outs = generator(prompts, max_new_tokens=60, do_sample=True, temperature=0.7, top_p=0.8, top_k=20)
for o in outs:
    print(o[0]["generated_text"][-1]["content"], "\n---")

<think>

</think>

Here’s a name for a pet robot that combines functionality and whimsy:

**"Pawsie"**  
*(A playful, friendly name that sounds both cute and robotic.)*

Or if you prefer something more techy:  

**"Nexa"**  
 
---
<think>

</think>

The sea's breath is calm,  
Dawn breaks on waves,  
And time is still. 
---


## 2. The `generate` API

Load the tokenizer and model yourself, build the prompt with `apply_chat_template`, and call `model.generate`.
This exposes Qwen3's `enable_thinking` switch and lets us split the `<think>...</think>` reasoning from the final answer.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
model.eval()

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

596M parameters


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): Qwe

In [15]:
def chat(messages, enable_thinking=True, max_new_tokens=512, **gen_kwargs):
    """Generate a reply and return (thinking, answer)."""
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, **gen_kwargs)

    # Keep only the newly generated tokens
    new_ids = output_ids[0][inputs.input_ids.shape[1]:].tolist()

    # Split at the last </think> token (if present)
    think_end_id = tokenizer.convert_tokens_to_ids("</think>")
    try:
        split = len(new_ids) - new_ids[::-1].index(think_end_id)
    except ValueError:
        split = 0

    thinking = tokenizer.decode(new_ids[:split], skip_special_tokens=True).strip()
    answer = tokenizer.decode(new_ids[split:], skip_special_tokens=True).strip()
    return thinking, answer

In [16]:
# Thinking mode: Qwen recommends temperature=0.6, top_p=0.95, top_k=20
thinking, answer = chat(
    [{"role": "user", "content": "What is 17 * 23? Show your work briefly."}],
    enable_thinking=True,
    do_sample=True, temperature=0.6, top_p=0.95, top_k=20,
)
print("THINKING:\n", thinking)
print("\nANSWER:\n", answer)

THINKING:
 

ANSWER:
 <think>
Okay, so I need to figure out what 17 multiplied by 23 is. Let me think. Hmm, I remember that when you multiply two numbers, you can break them down to make it easier. Maybe I can do this step by step. Let me try.

First, I know that 17 is 10 plus 7, right? And 23 is 20 plus 3. Maybe I can use the distributive property here. So, (10 + 7) * (20 + 3). If I distribute it, that would be 10*20 plus 10*3 plus 7*20 plus 7*3. Let me calculate each part.

10 times 20 is 200. Then 10 times 3 is 30. Next, 7 times 20 is 140, and 7 times 3 is 21. Adding all those together: 200 + 30 is 230, plus 140 is 370, and then plus 21 makes 391. Wait, so is the answer 391? Let me check with another method to be sure.

Alternatively, I can multiply 17 and 23 directly. Let me do that. 17 times 20 is 340, and 17 times 3 is 51. Adding those together: 340 + 51 equals 391. Okay, same result. So both methods give me 391. That seems right.

But just to make sure I didn't make any mistakes

In [17]:
# Non-thinking mode: Qwen recommends temperature=0.7, top_p=0.8, top_k=20
_, answer = chat(
    [{"role": "user", "content": "Name three uses of transformers outside NLP."}],
    enable_thinking=False,
    max_new_tokens=150,
    do_sample=True, temperature=0.7, top_p=0.8, top_k=20,
)
print(answer)

Transformers are a core component of NLP, but they have **no direct uses** outside of NLP. However, here are three **outside NLP** uses of transformers:

1. **Medical imaging** – Transformers are used to process and analyze medical images for tasks like radiology diagnosis, where they help with object recognition and segmentation.

2. **Robotics** – Transformers are used in robotics to process sensor data and make decisions in real-time, enabling autonomous navigation and object interaction.

3. **Natural language processing for general applications** – Transformers are foundational in building large-scale NLP systems that support chatbots, language translation, and text generation, which are used in various industries.


In [ ]:
# Multi-turn conversation: append the assistant's reply and continue
history = [{"role": "user", "content": "Pick a random fruit."}]
_, reply = chat(history, enable_thinking=False, max_new_tokens=50, do_sample=False)
print("Assistant:", reply)

history += [
    {"role": "assistant", "content": reply},
    {"role": "user", "content": "What color is it?"},
]
_, reply = chat(history, enable_thinking=False, max_new_tokens=50, do_sample=False)
print("Assistant:", reply)

### Streaming tokens as they're generated

In [20]:
from transformers import TextStreamer

prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Tell me a two-sentence story about a lighthouse."}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

with torch.no_grad():
    model.generate(**inputs, max_new_tokens=120, streamer=streamer, do_sample=True, temperature=0.7, top_p=0.8, top_k=20)

The lighthouse stood on a cliff, its beacon shining bright at night, guiding ships safely away from the storm. It was a beacon of hope, a reminder to sailors that they were safe and on the right path.


## 3. Next-token probabilities

`generate` only returns the token that was picked. To see the whole distribution, run a plain forward pass: the model returns **logits** (one score per vocabulary token) for every position. The logits at the *last* position score the next token, and `softmax` turns them into probabilities.

In [ ]:
def next_token_probs(prompt, top_k=10, temperature=1.0):
    """Return the top_k most likely next tokens as (token, prob) pairs, plus the full prob tensor."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits          # (batch, seq_len, vocab_size)
    next_logits = logits[0, -1, :]                # scores for the token after the prompt
    probs = torch.softmax(next_logits / temperature, dim=-1)

    top = torch.topk(probs, top_k)
    rows = [(tokenizer.decode(i), p.item()) for p, i in zip(top.values, top.indices)]
    return rows, probs

def show(rows):
    for tok, p in rows:
        print(f"{p:7.2%}  {tok!r}")

In [ ]:
import matplotlib.pyplot as plt

# Plot styling: one hue for a single series, neutral ink, recessive grid.
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e3e2de"

def style_ax(ax, grid_axis="x"):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED, length=0)
    ax.grid(axis=grid_axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)

def plot_top_probs(rows, title, ax, n_labels=3):
    """Horizontal bars for (token, prob) rows, most likely at the top; label the top few."""
    toks = [repr(t) for t, _ in rows][::-1]
    ps = [p for _, p in rows][::-1]
    ax.barh(toks, ps, height=0.6, color=BLUE)
    for y, p in list(enumerate(ps))[-n_labels:]:
        ax.text(p, y, f"  {p:.1%}", va="center", color=INK, fontsize=9)
    ax.set_title(title, loc="left", color=INK, fontsize=11)
    ax.set_xlabel("probability of next token", color=MUTED)
    style_ax(ax, "x")

In [ ]:
# Raw text continuation
rows, probs = next_token_probs("The capital of France is")
show(rows)
print(f"\nsum of all probs = {probs.sum().item():.4f}  (vocab size {probs.numel()})")

In [ ]:
# Top-10 next tokens as bars
rows, _ = next_token_probs("The capital of France is", top_k=10)
fig, ax = plt.subplots(figsize=(7, 4))
plot_top_probs(rows, 'Next token after "The capital of France is"', ax)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

**The long tail and `top_k` / `top_p`.** Sorting the whole vocabulary by probability shows how fast it falls off (left, log scale) and how much mass the top tokens hold (right). `top_k` cuts the list at a rank; `top_p` cuts it where the cumulative probability reaches p.

In [ ]:
_, probs = next_token_probs("The capital of France is")
sorted_p, _ = torch.sort(probs, descending=True)
N = 100
ranks = list(range(1, N + 1))
cum = torch.cumsum(sorted_p, dim=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

ax1.plot(ranks, sorted_p[:N].tolist(), color=BLUE, linewidth=2)
ax1.set_yscale("log")
ax1.set_xlabel("token rank", color=MUTED)
ax1.set_ylabel("probability (log scale)", color=MUTED)
ax1.set_title(f"Probability by rank (top {N} of {probs.numel():,})", loc="left", color=INK, fontsize=11)
style_ax(ax1, "y")

ax2.plot(ranks, cum[:N].tolist(), color=BLUE, linewidth=2)
for p_cut in (0.8, 0.95):
    k = int((cum >= p_cut).nonzero()[0].item()) + 1
    ax2.axhline(p_cut, color=MUTED, linewidth=0.8, linestyle="--")
    ax2.text(N, p_cut - 0.04, f"top_p={p_cut} keeps {k} token{'s' if k > 1 else ''}", ha="right", color=MUTED, fontsize=9)
ax2.set_ylim(0, 1.02)
ax2.set_xlabel("token rank", color=MUTED)
ax2.set_ylabel("cumulative probability", color=MUTED)
ax2.set_title("Cumulative probability", loc="left", color=INK, fontsize=11)
style_ax(ax2, "y")

plt.tight_layout()
plt.show()

In [ ]:
# Same idea with a chat prompt: the distribution over the first token of the assistant's reply
chat_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Is the sky blue? Answer yes or no."}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
rows, _ = next_token_probs(chat_prompt)
show(rows)

### Effect of temperature and entropy

Dividing the logits by `temperature` before the softmax sharpens (<1) or flattens (>1) the distribution. Entropy summarizes how uncertain the model is: low means confident, high means spread out.

In [ ]:
prompt = "Once upon a time, there was a"

for T in (0.3, 1.0, 2.0):
    rows, probs = next_token_probs(prompt, top_k=5, temperature=T)
    entropy = -(probs * torch.log(probs + 1e-12)).sum().item()
    print(f"temperature={T}  entropy={entropy:.2f} nats")
    show(rows)
    print()

In [ ]:
# Same prompt at three temperatures, shared x-axis so bar lengths are comparable
prompt = "Once upon a time, there was a"
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True)
for ax, T in zip(axes, (0.3, 1.0, 2.0)):
    rows, probs = next_token_probs(prompt, top_k=5, temperature=T)
    entropy = -(probs * torch.log(probs + 1e-12)).sum().item()
    plot_top_probs(rows, f"T = {T}  (entropy {entropy:.2f} nats)", ax, n_labels=2)
axes[0].set_xlim(0, 1)
plt.tight_layout()
plt.show()

### Probabilities of the tokens `generate` actually chose

Pass `output_scores=True` and `return_dict_in_generate=True`, then `compute_transition_scores` gives the log-probability of each generated token.

In [ ]:
prompt = "The three primary colors are"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=12, do_sample=False,
        output_scores=True, return_dict_in_generate=True,
    )

scores = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
new_ids = out.sequences[0, inputs.input_ids.shape[1]:]

for tok_id, logp in zip(new_ids, scores[0]):
    print(f"{logp.exp().item():7.2%}  {tokenizer.decode(tok_id)!r}")

In [ ]:
# Probability of each token greedy decoding picked: where was the model unsure?
tokens = [tokenizer.decode(t) for t in new_ids]
step_probs = scores[0].exp().tolist()

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(range(len(tokens)), step_probs, width=0.6, color=BLUE)
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels([repr(t) for t in tokens], rotation=45, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("probability of chosen token", color=MUTED)
ax.set_title(f'Greedy generation after "{prompt}"', loc="left", color=INK, fontsize=11)
style_ax(ax, "y")
plt.tight_layout()
plt.show()

## 4. Fine-tuning with Hugging Face: LoRA + `Trainer`

A general walkthrough of the standard fine-tuning workflow, using Qwen3-0.6B as a **sentiment classifier** on SST-2 movie-review snippets.

The pipeline:
1. **Data**: load a dataset with `datasets` and turn each example into `input_ids` + `labels`.
2. **Model**: load the pretrained model and wrap it with **LoRA** (via `peft`) so only a tiny set of adapter weights trains, which is what makes this feasible on a CPU.
3. **Train**: hand the model, arguments, dataset and a data collator to `Trainer`.
4. **Evaluate**: compare accuracy before and after, and look at how the next-token probabilities moved (the same idea as section 3).
5. **Save / reload** just the adapter.

The task is framed as text generation: the prompt is `Review: <text>\nSentiment:` and the model should continue with ` positive` or ` negative`. Because it is a causal LM, "classifying" means comparing the next-token logits of those two words.

Needs internet the first time (downloads SST-2). Training takes a few minutes on CPU.

### 1. Data

Each training example is the prompt followed by the label word and an end-of-sequence token. The `labels` copy the input but set the prompt positions to `-100`, which PyTorch's cross-entropy ignores, so the loss is computed only on the answer tokens.

In [ ]:
import time
from datasets import load_dataset

sst2 = load_dataset("glue", "sst2")
train_raw = sst2["train"].shuffle(seed=0).select(range(600))
eval_raw = sst2["validation"].shuffle(seed=0).select(range(200))
print(train_raw[0])

LABEL_WORDS = {0: " negative", 1: " positive"}
# The first token of each label word is what we compare at inference time
NEG_ID = tokenizer(LABEL_WORDS[0], add_special_tokens=False).input_ids[0]
POS_ID = tokenizer(LABEL_WORDS[1], add_special_tokens=False).input_ids[0]
print("label tokens:", repr(tokenizer.decode(NEG_ID)), repr(tokenizer.decode(POS_ID)))

def make_prompt(sentence):
    return f"Review: {sentence.strip()}\nSentiment:"

def tokenize_example(ex):
    prompt_ids = tokenizer(make_prompt(ex["sentence"]), add_special_tokens=False).input_ids
    answer_ids = tokenizer(LABEL_WORDS[ex["label"]], add_special_tokens=False).input_ids + [tokenizer.eos_token_id]
    return {
        "input_ids": prompt_ids + answer_ids,
        "labels": [-100] * len(prompt_ids) + answer_ids,
    }

train_ds = train_raw.map(tokenize_example, remove_columns=train_raw.column_names)
ex = train_ds[0]
print("\nprompt (masked):", repr(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l == -100])))
print("target (trained):", repr(tokenizer.decode([l for l in ex["labels"] if l != -100])))

### 2. Model + LoRA

LoRA freezes the pretrained weights and learns a low-rank update `W + (alpha/r) · B·A` for selected weight matrices. Here we target the attention `q_proj` and `v_proj` layers with rank `r=8`. We load a fresh copy of the model so the earlier sections' `model` stays untouched.

In [ ]:
from peft import LoraConfig, get_peft_model

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)
peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

### 3. Baseline: zero-shot accuracy

Before training, score the held-out reviews. For each one we run a forward pass and read the logits at the last prompt token: if ` positive` outscores ` negative`, we predict positive. LoRA's `B` matrices start at zero, so right now the wrapped model behaves exactly like the base model.

In [ ]:
@torch.no_grad()
def prob_positive(m, sentences, batch_size=8):
    # P(positive) from a two-way softmax over the " positive" / " negative" logits
    m.eval()
    tokenizer.padding_side = "right"
    out = []
    for i in range(0, len(sentences), batch_size):
        enc = tokenizer([make_prompt(s) for s in sentences[i:i + batch_size]],
                        return_tensors="pt", padding=True, add_special_tokens=False)
        logits = m(**enc).logits
        last = enc.attention_mask.sum(dim=1) - 1            # index of the last real token
        final = logits[torch.arange(len(last)), last]       # (batch, vocab)
        out.append(torch.softmax(final[:, [NEG_ID, POS_ID]], dim=-1)[:, 1])
    return torch.cat(out)

def accuracy(m, raw):
    p = prob_positive(m, raw["sentence"])
    return ((p > 0.5).long() == torch.tensor(raw["label"])).float().mean().item()

t0 = time.time()
acc_before = accuracy(peft_model, eval_raw)
print(f"accuracy before fine-tuning: {acc_before:.1%}  ({time.time() - t0:.0f}s)")

### 4. Train with `Trainer`

`Trainer` handles batching, the optimizer, the learning-rate schedule, gradient clipping and logging. We provide:
- `TrainingArguments`: hyperparameters and settings (`use_cpu=True` because we are on an Intel Mac)
- a **data collator** that pads each batch to its longest sequence (`DataCollatorForSeq2Seq` pads `labels` with `-100`, which is what we want for causal-LM fine-tuning)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="sst2-lora-out",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=5,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    use_cpu=True,
    dataloader_pin_memory=False,
    seed=0,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)

t0 = time.time()
trainer.train()
print(f"trained in {time.time() - t0:.0f}s")

In [ ]:
history = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
steps, losses = zip(*history)

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(steps, losses, color=BLUE, linewidth=2)
ax.set_xlabel("training step", color=MUTED)
ax.set_ylabel("loss", color=MUTED)
ax.set_title("Training loss", loc="left", color=INK, fontsize=11)
style_ax(ax, "y")
plt.tight_layout()
plt.show()

### 5. Evaluate after fine-tuning

`peft_model.disable_adapter()` temporarily turns LoRA off, which gives the exact base model back. Using it here means the before/after comparison below uses the same model object.

In [ ]:
t0 = time.time()
acc_after = accuracy(peft_model, eval_raw)
print(f"accuracy before: {acc_before:.1%}   after: {acc_after:.1%}  ({time.time() - t0:.0f}s)")

fig, ax = plt.subplots(figsize=(5, 3.6))
ax.bar(["base model", "LoRA fine-tuned"], [acc_before, acc_after], width=0.5, color=[BLUE, ORANGE])
for i, v in enumerate([acc_before, acc_after]):
    ax.text(i, v + 0.01, f"{v:.1%}", ha="center", color=INK, fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_ylabel(f"accuracy on {len(eval_raw)} held-out reviews", color=MUTED)
ax.set_title("SST-2 accuracy before vs. after", loc="left", color=INK, fontsize=11)
style_ax(ax, "y")
plt.tight_layout()
plt.show()

In [ ]:
# How did the model's confidence change on a few reviews?
examples = [
    "A moving, beautifully acted film that I loved from start to finish.",
    "Dull, predictable and far too long. I wanted to leave halfway through.",
    "It has a few good moments, but overall it never quite comes together.",
]
p_after = prob_positive(peft_model, examples)
with peft_model.disable_adapter():
    p_before = prob_positive(peft_model, examples)

print(f"{'P(positive) before':>20} {'after':>8}   review")
for s, pb, pa in zip(examples, p_before, p_after):
    print(f"{pb.item():>20.1%} {pa.item():>8.1%}   {s}")

### 6. Save and reload the adapter

`save_pretrained` writes only the adapter weights (a few MB), not the 0.6B base model. To use it later, load the base model and attach the adapter with `PeftModel.from_pretrained`. `merge_and_unload()` folds the adapter into the base weights if you want a plain `transformers` model for deployment.

In [ ]:
peft_model.save_pretrained("sst2-lora-adapter")

# --- later, in a fresh session ---
# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
# tuned = PeftModel.from_pretrained(base, "sst2-lora-adapter")
# merged = tuned.merge_and_unload()   # optional: plain model with the LoRA folded in